In [ ]:
import json
import boto3
import requests
from bs4 import BeautifulSoup
import datetime
import os

In [ ]:
S3_BUCKET = os.getenv("S3_BUCKET")
FIXTURE_KEY = "fixtures.json"
RAW_PREFIX = os.getenv("RAW_PREFIX")
# BRONZE_PREFIX = os.getenv("BRONZE_PREFIX")

In [ ]:
s3 = boto3.client("s3")

In [ ]:
def fetch_schedule():
    """Reads match schedule from S3 and returns a list of match URLs."""
    try:
        response = s3.get_object(Bucket=S3_BUCKET, Key=f"{RAW_PREFIX}{FIXTURE_KEY}")
        match_schedule = json.loads(response["Body"].read().decode("utf-8"))
        return {match["link"]: match["match"] for match in match_schedule if "link" in match}
    except Exception as e:
        print(f"Error reading match schedule: {e}")
        return {}

In [ ]:
def get_existing_data(match_name):
    """Fetches existing match data from S3 if available."""
    try:
        file_key = f"{RAW_PREFIX}{match_name.replace(' ', '_')}.json"
        response = s3.get_object(Bucket=S3_BUCKET, Key=file_key)
        existing_data = json.loads(response["Body"].read().decode("utf-8"))
        return existing_data
    except s3.exceptions.NoSuchKey:
        return []  # No previous data, return empty list
    except Exception as e:
        print(f"Error fetching existing data for {match_name}: {e}")
        return []

In [ ]:
def scrape_match_data(match_url, last_over=None, last_ball=None):
    """Scrapes new ball-by-ball data from a match URL."""
    try:
        response = requests.get(match_url, headers={"User-Agent": "Mozilla/5.0"})
        if response.status_code != 200:
            raise Exception(f"Failed to fetch data, status code: {response.status_code}")

        soup = BeautifulSoup(response.text, "html.parser")
        match_title = soup.find("h1", class_="cb-nav-hdr").text.strip() if soup.find("h1", class_="cb-nav-hdr") else "Unknown Match"
        balls_bowled = soup.find_all("div", class_="cb-col cb-col-100 ng-scope")

        new_data = []
        for ball in reversed(balls_bowled):  
            over_info = ball.find("div", class_="cb-mat-mnu-wrp cb-ovr-num ng-binding ng-scope")
            if not over_info:
                continue
            
            over, ball_no = map(int, over_info.text.strip().split("."))

            # Skip balls that were already extracted
            if last_over is not None and (over < last_over or (over == last_over and ball_no <= last_ball)):
                continue

            event_info = ball.find("p", class_="cb-com-ln").text.strip() if ball.find("p", class_="cb-com-ln") else "N/A"
            
            new_data.append({
                "match": match_title,
                "over": over,
                "ball": ball_no,
                "event": event_info
            })

        return new_data

    except Exception as e:
        print(f"Error scraping {match_url}: {e}")
        return []

In [ ]:
def lambda_handler(event, context):
    match_urls = fetch_schedule()
    
    if not match_urls:
        return {"statusCode": 500, "body": "No matches found in schedule"}

    for match_url, match_name in match_urls.items():
        existing_data = get_existing_data(match_name)

        # Find last extracted over and ball
        last_over, last_ball = None, None
        if existing_data:
            last_extracted = existing_data[-1]
            last_over, last_ball = last_extracted["over"], last_extracted["ball"]

        # Scrape only new balls
        new_data = scrape_match_data(match_url, last_over, last_ball)

        if new_data:
            updated_data = existing_data + new_data  # Append new data to existing
            file_name = f"{S3_KEY_PREFIX}{match_name.replace(' ', '_')}.json"
            s3.put_object(Bucket=S3_BUCKET, Key=file_name, Body=json.dumps(updated_data))

    return {"statusCode": 200, "body": "New data extracted and stored in S3"}